# 00 – Setup check

**Goal:** Verify that the environment is ready: Python packages, PostgreSQL connection and database schema.
**Input:** none · **Output:** none (diagnostics only)
**Assessment items:** P3 (database with SQL queries from Python), B2 (PostgreSQL), B6 (public GitHub repo)

In [ ]:
import sys
from pathlib import Path

# Make the repository root importable, whether Jupyter starts in the root or in notebooks/
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)

## 1. Python packages

In [ ]:
from importlib.metadata import version

packages = ["fastf1", "pandas", "numpy", "sqlalchemy", "psycopg", "statsmodels",
            "scikit-learn", "xgboost-cpu", "shap", "geopandas", "folium", "rapidfuzz"]
print(f"Python {sys.version.split()[0]}")
for pkg in packages:
    print(f"  {pkg:<14} {version(pkg)}")

## 2. Database connection

In [ ]:
import pandas as pd
from sqlalchemy import text
from src.db import get_engine, init_schema, table_counts

engine = get_engine()
with engine.connect() as conn:
    print(conn.execute(text("SELECT version()")).scalar_one())

## 3. Schema

`init_schema` is idempotent: it creates missing tables and leaves existing data untouched.

In [ ]:
tables = init_schema(engine)
pd.Series(table_counts(engine), name="rows").to_frame()

In [ ]:
# Column overview straight from the PostgreSQL catalog
pd.read_sql(
    """
    SELECT table_name, COUNT(*) AS n_columns,
           STRING_AGG(column_name, ', ' ORDER BY ordinal_position) AS columns
    FROM information_schema.columns
    WHERE table_schema = 'public'
    GROUP BY table_name
    ORDER BY table_name
    """,
    engine,
)